# Notebook 01 — Bronze Ingestion
**Project:** Card Transaction Fraud Monitoring  
**Platform:** Databricks + PySpark + Unity Catalog + Delta Lake + Auto Loader

## Objective
Ingest the six supplied CSV datasets from the Unity Catalog volume into six Bronze Delta tables.

### Design decisions
- Use Auto Loader (`cloudFiles`) to process existing files and new arrivals incrementally.
- Read all 11 transaction batch files from the transaction folder into one table.
- Keep original CSV values as strings in Bronze; casting and cleaning belong in Silver.
- Add source-file and ingestion-timestamp metadata.
- Use a separate schema location and checkpoint for each source.
- Use `availableNow=True` so the job processes available files and then stops.

**Important:** Do not delete or casually change checkpoint directories between runs.

## 1. Import libraries

In [0]:
from pyspark.sql.functions import col, current_timestamp

## 2. Configure catalog, schema and volume paths

The source volume is `capstone_team01.bronze_schema.raw_data`. CSV datasets are under its `datasets` directory. Bronze output tables will be created in `capstone_team01.bronze_schema` (for example `capstone_team01.bronze_schema.bronze_card_transactions`).

In [0]:
CATALOG = "capstone_team01"
BRONZE_SCHEMA = "bronze_schema"

VOLUME_ROOT = "/Volumes/capstone_team01/bronze_schema/raw_data"
VOLUME_PATH = f"{VOLUME_ROOT}/datasets"

CHECKPOINT_PATH = f"{VOLUME_ROOT}/_checkpoints"
SCHEMA_PATH = f"{VOLUME_ROOT}/_schemas"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

print("Catalog:", CATALOG)
print("Bronze schema:", BRONZE_SCHEMA)
print("Source path:", VOLUME_PATH)

## 3. Verify source folders before ingestion

Run this cell first. Confirm that the six expected source locations are present before starting ingestion.

In [0]:
display(dbutils.fs.ls(VOLUME_PATH))

## 4. Define the six source paths

These paths follow the folder structure supplied for the capstone datasets.

In [0]:
sources = {
    "card_transactions":
        f"{VOLUME_PATH}/transaction_data/card_transactions",

    "payment_status_changes":
        f"{VOLUME_PATH}/cdc_data/payment_status_changes.csv",

    "device_authentication_events":
        f"{VOLUME_PATH}/event_data/device_authentication_events.csv",

    "card_master":
        f"{VOLUME_PATH}/master_data/azure_sql_seed/card_master.csv",

    "customer_master":
        f"{VOLUME_PATH}/master_data/azure_sql_seed/customer_master.csv",

    "merchant_master":
        f"{VOLUME_PATH}/master_data/azure_sql_seed/merchant_master.csv"
}

for name, path in sources.items():
    print(f"{name}: {path}")

## 5. Create metadata directories

These directories store Auto Loader schema information and streaming checkpoints. They are separate from the source CSV files.

In [0]:
dbutils.fs.mkdirs(CHECKPOINT_PATH)
dbutils.fs.mkdirs(SCHEMA_PATH)

print("Checkpoint and schema directories are ready.")

## 6. Define a reusable Auto Loader ingestion function

The function reads CSV files, keeps source columns as strings, adds source metadata, and writes append-only records to a Bronze Delta table.

In [0]:
def ingest_to_bronze(source_name, source_path):

    schema_location = f"{SCHEMA_PATH}/{source_name}"
    checkpoint_location = f"{CHECKPOINT_PATH}/{source_name}"
    table_name = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_{source_name}"

    reader = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.inferColumnTypes", "false")
        .option("header", "true")
    )

    # Auto Loader fails to infer a schema from a single-file path on this compute.
    # For a single CSV file, read its parent folder and keep only that file name.
    if source_path.endswith(".csv"):
        folder_path = source_path.rsplit("/", 1)[0]
        file_name = source_path.rsplit("/", 1)[1]
        df = reader.option("pathGlobFilter", file_name).load(folder_path)
    else:
        df = reader.load(source_path)

    df = (
        df.withColumn("source_file", col("_metadata.file_path"))
          .withColumn("ingestion_timestamp", current_timestamp())
    )

    query = (
        df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_location)
        .trigger(availableNow=True)
        .toTable(table_name)
    )

    query.awaitTermination()
    print(f"Completed: {table_name}")

## 7. Ingest all six datasets

The transaction folder contains multiple batch CSV files. Auto Loader reads the folder and writes them together into one Bronze transaction table.

In [0]:
for source_name, source_path in sources.items():
    print(f"Starting ingestion: {source_name}")
    ingest_to_bronze(source_name, source_path)

## 8. Verify Bronze tables

Expected baseline counts from the capstone source document are shown for comparison. Actual counts may differ if source files have changed.

In [0]:
expected_counts = {
    "bronze_card_transactions": 10689,
    "bronze_payment_status_changes": 259,
    "bronze_device_authentication_events": 541,
    "bronze_card_master": 124,
    "bronze_customer_master": 106,
    "bronze_merchant_master": 115
}

for table_name, expected_count in expected_counts.items():

    full_table_name = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    actual_count = spark.table(full_table_name).count()

    print(
        f"{table_name} | Expected: {expected_count} | "
        f"Actual: {actual_count} | Match: {actual_count == expected_count}"
    )

## 9. Inspect sample records

In [0]:
for source_name in sources.keys():

    table_name = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_{source_name}"

    print(f"Sample records: {table_name}")
    display(spark.table(table_name).limit(5))

## 10. List created Bronze tables

In [0]:
spark.sql(f"SHOW TABLES IN {CATALOG}.{BRONZE_SCHEMA}").show(truncate=False)

## Completion checklist

- [ ] Source volume path verified.
- [ ] All six source paths verified.
- [ ] Auto Loader schema and checkpoint directories created.
- [ ] Six Bronze Delta tables created.
- [ ] Actual row counts compared with source-document baseline.
- [ ] Sample records and ingestion metadata inspected.

### Notes
- No deduplication, business validation, type casting or standardization is performed in Bronze.
- Do not reuse a checkpoint across different datasets.
- If an ingestion query fails, inspect the error before rerunning. Do not delete its checkpoint as a first troubleshooting step.
- If the volume does not permit writing `_checkpoints` and `_schemas`, configure a separate writable volume for Auto Loader metadata.
- Azure SQL integration remains separate. For this baseline ingestion, the supplied master-data CSV extracts are used.